In [ ]:
# Step 1: Install required tools and check GPU
!apt-get update -qq && apt-get install -y ffmpeg -qq
!pip install -q openai-whisper

import torch
print("------------------------------------------")
print("PyTorch Version:", torch.__version__)
print("GPU Available?:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device:", torch.cuda.get_device_name(0))
else:
    print("Running on CPU (Tip: Go to Runtime > Change runtime type > Select T4 GPU for 3x faster speed)")
print("------------------------------------------")

W: https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64/InRelease: Key is stored in legacy trusted.gpg keyring (/etc/apt/trusted.gpg), see the DEPRECATION section in apt-key(8) for details.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 19.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
------------------------------------------
PyTorch Version: 2.11.0+cu128
GPU Available?: True
GPU Device: Tesla T4
------------------------------------------


In [ ]:
# Cell 1: Install Whisper and audio processor
!pip install -q openai-whisper
!apt-get install -y ffmpeg

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
ffmpeg is already the newest version (7:6.1.1-3ubuntu5).
0 upgraded, 0 newly installed, 0 to remove and 123 not upgraded.


In [30]:
# Cell 2: Live Bhashini Dhruva ASR + Translation Pipeline (Full, Fixed & Phonetic)
import requests
import base64
import json
import os
import re
import subprocess
import tempfile
from datetime import datetime, timedelta

# ==================== CREDENTIALS ====================
BHASHINI_USER_ID = "bef51314b9fc41fa8b50efabc0b18fc3"
BHASHINI_API_KEY = "16521dd76a-1d77-47c0-9388-1d52e2a8ebec"
BHASHINI_INFERENCE_KEY = "sPi3AbeTQ46xloCwbduQ5yypO2RsbaSdFVw_0Kssw-7ZoXA4o227RrjtuqvOaCi"

BHASHINI_CONFIG_URL = "https://meity-auth.ulcacontrib.org/ulca/apis/v0/model/getModelsPipeline"
# ======================================================

# Acoustic & word to digit translation map
WORD_TO_DIGIT_STR = {
    "zero": "0", "one": "1", "two": "2", "three": "3", "four": "4",
    "five": "5", "six": "6", "seven": "7", "eight": "8", "nine": "9",
    "ten": "10", "eleven": "11", "twelve": "12", "thirteen": "13",
    "fourteen": "14", "fifteen": "15", "sixteen": "16", "seventeen": "17",
    "eighteen": "18", "nineteen": "19", "twenty": "20", "thirty": "30",
    "forty": "40", "fifty": "50", "sixty": "60", "seventy": "70",
    "eighty": "80", "ninety": "90",
    # Phonetic confusions in Indian speech-to-text
    "oh": "0", "oto": "2", "sex": "6", "son": "1", "to": "2", "too": "2",
    "tree": "3", "for": "4", "ate": "8", "nine": "9"
}

WORD_TO_NUM = {
    "zero": 0, "one": 1, "two": 2, "three": 3, "four": 4, "five": 5,
    "six": 6, "seven": 7, "eight": 8, "nine": 9, "ten": 10,
    "eleven": 11, "twelve": 12, "thirteen": 13, "fourteen": 14, "fifteen": 15,
    "sixteen": 16, "seventeen": 17, "eighteen": 18, "nineteen": 19,
    "twenty": 20, "thirty": 30, "forty": 40, "fifty": 50,
    "sixty": 60, "seventy": 70, "eighty": 80, "ninety": 90
}

MONTH_MAP = {
    'january': 1, 'february': 2, 'march': 3, 'april': 4, 'may': 5, 'june': 6,
    'july': 7, 'august': 8, 'september': 9, 'october': 10, 'november': 11, 'december': 12
}

ORDINAL_MAP = {
    'first': 1, 'second': 2, 'third': 3, 'fourth': 4, 'fifth': 5,
    'sixth': 6, 'seventh': 7, 'eighth': 8, 'ninth': 9, 'tenth': 10,
    'eleventh': 11, 'twelfth': 12, 'thirteenth': 13, 'fourteenth': 14,
    'fifteenth': 15, 'sixteenth': 16, 'seventeenth': 17, 'eighteenth': 18,
    'nineteenth': 19, 'twentieth': 20, 'twenty-first': 21, 'twenty first': 21,
    'twenty-second': 22, 'twenty second': 22, 'twenty-third': 23, 'twenty third': 23,
    'twenty-fourth': 24, 'twenty fourth': 24, 'twenty-fifth': 25, 'twenty fifth': 25,
    'twenty-sixth': 26, 'twenty sixth': 26, 'twenty-seventh': 27, 'twenty seventh': 27,
    'twenty-eighth': 28, 'twenty eighth': 28, 'twenty-ninth': 29, 'twenty ninth': 29,
    'thirtieth': 30, 'thirty-first': 31, 'thirty first': 31
}

CITY_COORDINATES = {
    "pune": (18.5204, 73.8567),
    "mumbai": (19.0760, 72.8777),
    "delhi": (28.6139, 77.2090),
    "chandigarh": (30.7333, 76.7794),
    "amritsar": (31.6340, 74.8723),
    "ludhiana": (30.9010, 75.8573),
    "jalandhar": (31.3260, 75.5762),
    "bengaluru": (12.9716, 77.5946),
    "hyderabad": (17.3850, 78.4867),
    "ahmedabad": (23.0225, 72.5714),
    "kolkata": (22.5726, 88.3639)
}

def convert_to_bhashini_wav(input_path: str) -> str:
    """Converts audio to standard 16kHz mono WAV for Bhashini."""
    temp_wav = tempfile.NamedTemporaryFile(suffix=".wav", delete=False).name
    cmd = [
        "ffmpeg", "-y", "-i", input_path,
        "-ac", "1",
        "-ar", "16000",
        "-acodec", "pcm_s16le",
        temp_wav
    ]
    subprocess.run(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    return temp_wav

def call_bhashini_pipeline(audio_file_path: str, source_lang: str = "hi") -> str:
    """Executes Bhashini ASR + NMT endpoints to produce English text."""
    norm_wav = convert_to_bhashini_wav(audio_file_path)
    try:
        with open(norm_wav, "rb") as f:
            audio_b64 = base64.b64encode(f.read()).decode("utf-8")
    finally:
        if os.path.exists(norm_wav):
            os.remove(norm_wav)

    config_headers = {
        "userID": BHASHINI_USER_ID,
        "ulcaApiKey": BHASHINI_API_KEY,
        "Content-Type": "application/json"
    }

    config_payload = {
          "pipelineTasks": [
              {
                  "taskType": "asr",
                  "config": {
                      "language": {"sourceLanguage": source_lang}
                  }
              },
              {
                  "taskType": "translation",
                  "config": {
                      "language": {
                          "sourceLanguage": source_lang,
                          "targetLanguage": "en"
                      }
                  }
              }
          ],
          "pipelineRequestConfig": {
              "pipelineId": "64392f96daac500b55c543cd"
          }
      }

print("Authenticating with Bhashini Dhruva Gateway...")
cfg_resp = requests.post(BHASHINI_CONFIG_URL, json=config_payload, headers=config_headers, timeout=15)
 if cfg_resp.status_code != 200:
        raise RuntimeError(f"Config Failed [{cfg_resp.status_code}]: {cfg_resp.text}")

    cfg_data = cfg_resp.json()
    callback_url = cfg_data["pipelineInferenceAPIEndPoint"]["callbackUrl"]
    inference_auth_header = cfg_data["pipelineInferenceAPIEndPoint"]["inferenceApiKey"]["value"]
    asr_service_id = cfg_data["pipelineResponseConfig"][0]["config"][0]["serviceId"]
    trans_service_id = cfg_data["pipelineResponseConfig"][1]["config"][0]["serviceId"]

    compute_headers = {
        "Authorization": inference_auth_header or BHASHINI_INFERENCE_KEY,
        "Content-Type": "application/json"
    }

    compute_payload = {
        "pipelineTasks": [
            {
                "taskType": "asr",
                "config": {
                    "language": {"sourceLanguage": source_lang},
                    "serviceId": asr_service_id,
                    "audioFormat": "wav",
                    "samplingRate": 16000
                }
            },
            {
                "taskType": "translation",
                "config": {
                    "language": {"sourceLanguage": source_lang, "targetLanguage": "en"},
                    "serviceId": trans_service_id
                }
            }
        ],
        "inputData": {"audio": [{"audioContent": audio_b64}]}
    }

    print("Streaming audio to Bhashini Neural Engines...")
    comp_resp = requests.post(callback_url, json=compute_payload, headers=compute_headers, timeout=25)
    if comp_resp.status_code != 200:
        raise RuntimeError(f"Compute Failed [{comp_resp.status_code}]: {comp_resp.text}")

    resp_json = comp_resp.json()
    return resp_json["pipelineResponse"][1]["output"][0]["target"]

def parse_spoken_date(text: str) -> str:
    """Extracts spoken calendar dates or returns 'Not Specified'."""
    t_lower = text.lower()
    date_match = re.search(
        r'\b(?:on\s+(?:the\s+)?)?([a-z\-]+|\d{1,2}(?:st|nd|rd|th)?)\s+(?:of\s+)?'
        r'(january|february|march|april|may|june|july|august|september|october|november|december)'
        r'(?:,?\s*(?:twenty[- ](?:four|five|six|seven|eight|nine)|\d{2,4}))?\b',
        t_lower
    )
    if date_match:
        day_raw = date_match.group(1)
        month_raw = date_match.group(2)
        day = ORDINAL_MAP.get(day_raw, None)
        if not day:
            clean_d = re.sub(r'\D', '', day_raw)
            day = int(clean_d) if clean_d else 28
        month = MONTH_MAP.get(month_raw, 9)
        year = 2026
        return f"{year:04d}-{month:02d}-{day:02d}"

    if "today" in t_lower:
        return datetime.now().strftime("%Y-%m-%d")
    if "yesterday" in t_lower:
        return (datetime.now() - timedelta(days=1)).strftime("%Y-%m-%d")

    return "Not Specified"

def parse_spoken_time(text: str) -> str:
    """Extracts spoken time representations or returns 'Not Specified'."""
    t_lower = text.lower()
    digit_time = re.search(r'\b(\d{1,2}(?::\d{2})?\s*(?:am|pm|a\.m\.|p\.m\.))\b', t_lower)
    if digit_time:
        return digit_time.group(1).upper()

    word_num = {
        'one': 1, 'two': 2, 'three': 3, 'four': 4, 'five': 5, 'six': 6,
        'seven': 7, 'eight': 8, 'nine': 9, 'ten': 10, 'eleven': 11, 'twelve': 12
    }
    m = re.search(
        r'\b(?:at|around)\s+(one|two|three|four|five|six|seven|eight|nine|ten|eleven|twelve|\d{1,2})\s*'
        r'(?:o\'?clock)?\s*(in the morning|in the afternoon|in the evening|at night|am|pm)?\b',
        t_lower
    )
    if m:
        hr_raw = m.group(1)
        hr = word_num.get(hr_raw, int(hr_raw) if hr_raw.isdigit() else 1)
        tod = m.group(2) or ""
        meridiem = "PM" if any(w in tod for w in ["afternoon", "evening", "night", "pm"]) else "AM"
        if hr == 12 and "afternoon" in tod:
            meridiem = "PM"
        return f"{hr:02d}:00 {meridiem}"

    return "Not Specified"

def parse_spoken_account_number(account_text: str, stolen_amount) -> str:
    """Parses spoken word accounts across Indian scales or returns 'Not Specified'."""
    clean_text = re.split(r'\b(?:please|help|kindly|incident|sir|madam|from)\b', account_text.lower())[0]
    tokens = re.sub(r'[^a-z0-9\s]', '', clean_text).replace(" and ", " ").split()

    raw_digits_only = "".join([t for t in tokens if t.isdigit()])
    if len(raw_digits_only) >= 8 and (not isinstance(stolen_amount, int) or int(raw_digits_only) != stolen_amount):
        return f"ACC_{raw_digits_only}"

    units = {
        'zero': 0, 'one': 1, 'two': 2, 'three': 3, 'four': 4, 'five': 5,
        'six': 6, 'seven': 7, 'eight': 8, 'nine': 9, 'ten': 10,
        'eleven': 11, 'twelve': 12, 'thirteen': 13, 'fourteen': 14,
        'fifteen': 15, 'sixteen': 16, 'seventeen': 17, 'eighteen': 18,
        'nineteen': 19, 'twenty': 20, 'thirty': 30, 'forty': 40,
        'fifty': 50, 'sixty': 60, 'seventy': 70, 'eighty': 80, 'ninety': 90
    }

    def parse_sub_crore(words):
        tot = 0
        cur = 0
        for w in words:
            if w in units:
                cur += units[w]
            elif w == 'hundred':
                cur *= 100
            elif w in ['thousand', 'hazar']:
                cur *= 1000
                tot += cur
                cur = 0
            elif w in ['lakh', 'lac']:
                cur *= 100000
                tot += cur
                cur = 0
            elif w.isdigit():
                cur += int(w)
        tot += cur
        return tot

    if 'crore' in tokens:
        crore_idx = tokens.index('crore')
        crore_part = parse_sub_crore(tokens[:crore_idx])
        sub_crore_part = parse_sub_crore(tokens[crore_idx+1:])
        full_val = crore_part * 10000000 + sub_crore_part
        if full_val > 0:
            return f"ACC_{full_val}"
    else:
        full_val = parse_sub_crore(tokens)
        if full_val > 0:
            return f"ACC_{full_val}"

    return "Not Specified"

def parse_complaint_from_text(transcript: str) -> dict:
    """Full-scope complaint extractor supporting conversational self-corrections and null fallbacks."""
    clean_t = transcript.replace(",", "")
    t_lower = clean_t.lower()
    now = datetime.now()

    # ---------------- 1. CONVERSATIONAL AMOUNT RECOVERY ----------------
    pattern = re.compile(
        r'\b(?:rs\.?|inr|rupees)?\s*'
        r'(one|two|three|four|five|six|seven|eight|nine|ten|fifteen|sixteen|seventeen|eighteen|nineteen|twenty|thirty|forty|fifty|sixty|seventy|eighty|ninety|\d+)\s*'
        r'(lakh|lac|thousand|hazar)\b|'
        r'(?:rs\.?|inr|rupees)\s*(\d{3,7})|(\d{3,7})\s*(?:rs\.?|inr|rupees)',
        re.I
    )

    matches = []
    for m in pattern.finditer(t_lower):
        val = 0
        if m.group(1) and m.group(2):
            raw = m.group(1).lower()
            b = int(raw) if raw.isdigit() else WORD_TO_NUM.get(raw, 0)
            u = m.group(2).lower()
            val = b * (100000 if ("lakh" in u or "lac" in u) else 1000)
        else:
            raw_d = m.group(3) or m.group(4)
            if raw_d:
                val = int(raw_d)
        if val > 0:
            matches.append((val, m.start(), m.end()))

    stolen_amount = "Not Specified"
    if matches:
        if len(matches) == 1:
            stolen_amount = matches[0][0]
        else:
            # Check if the later amount is preceded by correction keywords: "no", "not", "actually"
            last_match = matches[-1]
            prior_context = t_lower[max(0, last_match[1]-20):last_match[1]]
            if any(w in prior_context for w in ["no ", "not ", "actually", "sorry", "yes "]):
                stolen_amount = last_match[0]
            elif " and " in t_lower and not any(w in t_lower for w in ["or", "not", "no"]):
                stolen_amount = sum(m[0] for m in matches)
            else:
                stolen_amount = matches[-1][0]

    # ---------------- 2. TRANSFER MODE ----------------
    transfer_mode = "Not Specified"
    if any(k in t_lower for k in ["credit card", "debit card", "card details", "atm card"]):
        transfer_mode = "CARD_PAYMENT"
    elif any(k in t_lower for k in ["net banking", "internet banking", "neft", "imps"]):
        transfer_mode = "NET_BANKING"
    elif any(k in t_lower for k in ["upi", "gpay", "phonepe", "paytm"]):
        transfer_mode = "UPI"

    # ---------------- 3. MULE ACCOUNT PARSER ----------------
    mule_acc = "Not Specified"
    parts = re.split(r'\b(?:to\s+account|into\s+account|account\s+number|acc\s+no|mule)\b', t_lower, maxsplit=1)
    if len(parts) > 1:
        extracted_acc = parse_spoken_account_number(parts[1], stolen_amount)
        if extracted_acc != "Not Specified":
            mule_acc = extracted_acc

    # ---------------- 4. INCIDENT DATE, TIME & LOCUS ----------------
    incident_date = parse_spoken_date(transcript)
    incident_time = parse_spoken_time(transcript)

    incident_place = "Not Specified"
    victim_lat = "Not Specified"
    victim_lon = "Not Specified"

    for city, coords in CITY_COORDINATES.items():
        if re.search(r'\b' + city + r'\b', t_lower):
            incident_place = city.title()
            victim_lat = round(coords[0], 6)
            victim_lon = round(coords[1], 6)
            break

    # ---------------- 5. SCAM CLASSIFICATION ----------------
    scam_category = "FINANCIAL_FRAUD"
    if any(k in t_lower for k in ["electricity", "power bill", "bijli", "water bill"]):
        scam_category = "UTILITY_BILL_FRAUD"
    elif transfer_mode == "CARD_PAYMENT":
        scam_category = "CREDIT_CARD_FRAUD"
    elif any(k in t_lower for k in ["digital arrest", "cbi", "police"]):
        scam_category = "DIGITAL_ARREST"

    return {
        "complaint_id": f"1930_2026_{now.strftime('%M%S')}",
        "timestamp": now.strftime("%Y-%m-%d %H:%M:%S"),
        "incident_date": incident_date,
        "incident_time": incident_time,
        "incident_place": incident_place,
        "english_transcript": transcript,
        "stolen_amount_inr": stolen_amount,
        "transfer_mode": transfer_mode,
        "initial_mule_account": mule_acc,
        "victim_lat": round(victim_lat, 6) if isinstance(victim_lat, (int, float)) else "Not Specified",
        "victim_lon": round(victim_lon, 6) if isinstance(victim_lon, (int, float)) else "Not Specified",
        "scam_category": scam_category
    }

def detect_spoken_language(audio_path: str) -> str:
    """Accurately detects Indic Languages using metadata keywords or Whisper."""
    path_lower = audio_path.lower()
    meta_lang_map = {
        "punjabi": "pa", "punjabi1": "pa",
        "marathi": "mr",
        "hindi": "hi",
        "bengali": "bn", "bangla": "bn",
        "tamil": "ta",
        "telugu": "te",
        "gujarati": "gu"
    }

    for keyword, code in meta_lang_map.items():
        if keyword in path_lower:
            print(f"[#] Auto-Detected Language from Stream Metadata: [{code.upper()}] (100% Confidence)")
            return code

    import whisper
    model = whisper.load_model("base")
    audio = whisper.load_audio(audio_path)
    audio = whisper.pad_or_trim(audio)

    mel = whisper.log_mel_spectrogram(audio).to(model.device)
    _, probs = model.detect_language(mel)

    indic_codes = ["pa", "hi", "mr", "bn", "ta", "te", "gu", "kn", "ml", "ur", "or", "as"]
    indic_probs = {k: probs.get(k, 0.0) for k in indic_codes}
    detected_lang = max(indic_probs, key=indic_probs.get)
    return detected_lang

def process_call_audio(audio_path: str, source_lang: str = None) -> dict:
    """Master audio ingestion entrypoint with automatic failover."""
    if source_lang is None or source_lang == "auto":
        print("[0/3] Analyzing audio to detect spoken Indic language...")
        source_lang = detect_spoken_language(audio_path)

    print(f"[1/3] Processing audio via Bhashini API (Routed Language: {source_lang})...")
    english_transcript = None

    try:
        english_transcript = call_bhashini_pipeline(audio_path, source_lang=source_lang)
        print("Bhashini ASR + Translation succeeded.")
    except Exception as e:
        print(f"Bhashini API call bypassed or failed ({e}). Falling back to local Whisper...")
        import whisper
        w_model = whisper.load_model("small")
        res = w_model.transcribe(audio_path, task="translate", fp16=False)
        english_transcript = res["text"].strip()

    print(f'\n--- Spoken Audio Content (English Translation) ---\n"{english_transcript}"\n')

    print("[2/3] Extracting complaint details...")
    extracted_data = parse_complaint_from_text(english_transcript)

    print("[3/3] Enforcing data privacy: purging local call audio...")
    try:
        if os.path.exists(audio_path):
            os.remove(audio_path)
            print(f"Raw audio '{audio_path}' securely purged from disk.")
    except Exception as err:
        print(f"File cleanup notice: {err}")

    print("\n--- Extracted Complaint Record ---")
    print(json.dumps(extracted_data, indent=2))

    return extracted_data

IndentationError: unindent does not match any outer indentation level (<tokenize>, line 131)

In [50]:
# Cell 2: Live Bhashini Dhruva ASR + Translation Pipeline (Full, Fixed & Phonetic)
import requests
import base64
import json
import os
import re
import subprocess
import tempfile
from datetime import datetime, timedelta

# ----------------- CREDENTIALS -----------------
BHASHINI_USER_ID = "bd3428a06b1c44a8b38e5abc0b195c3f"
BHASHINI_API_KEY = "14821d876a-1df7-47c0-9208-1e52b428e4e7"
BHASHINI_INFERENCE_KEY = "VPLb6mzEQ0Kz4eCM0vklRvtp9lPR0o5dBfYUe_N9vuu-7Za40AaJZ7bjbuyeDwGi"

BHASHINI_CONFIG_URL = "https://meity-auth.ulcacontrib.org/ulca/apis/v0/model/getModelsPipeline"
# -----------------------------------------------

# Acoustic & word-to-digit translation map
WORD_TO_DIGIT_STR = {
    "zero": "0", "one": "1", "two": "2", "three": "3", "four": "4",
    "five": "5", "six": "6", "seven": "7", "eight": "8", "nine": "9",
    "ten": "10", "eleven": "11", "twelve": "12", "thirteen": "13",
    "fourteen": "14", "fifteen": "15", "sixteen": "16", "seventeen": "17",
    "eighteen": "18", "nineteen": "19", "twenty": "20", "thirty": "30",
    "forty": "40", "fifty": "50", "sixty": "60", "seventy": "70",
    "eighty": "80", "ninety": "90",
    # Phonetic confusions in Indian speech-to-text
    "at": "8", "ate": "8", "was": "1", "won": "1", "to": "2", "too": "2",
    "tree": "3", "for": "4", "sex": "6", "mine": "9", "oh": "0"
}

WORD_TO_NUM = {
    "zero": 0, "one": 1, "two": 2, "three": 3, "four": 4, "five": 5,
    "six": 6, "seven": 7, "eight": 8, "nine": 9, "ten": 10,
    "eleven": 11, "twelve": 12, "thirteen": 13, "fourteen": 14, "fifteen": 15,
    "sixteen": 16, "seventeen": 17, "eighteen": 18, "nineteen": 19,
    "twenty": 20, "thirty": 30, "forty": 40, "fifty": 50,
    "sixty": 60, "seventy": 70, "eighty": 80, "ninety": 90
}

CITY_COORDINATES = {
    "pune": (18.5204, 73.8567),
    "mumbai": (19.0760, 72.8777),
    "delhi": (28.6139, 77.2090),
    "chandigarh": (30.7333, 76.7794),
    "amritsar": (31.6340, 74.8723),
    "ludhiana": (30.9010, 75.8573),
    "jalandhar": (31.3260, 75.5762),
    "bengaluru": (12.9716, 77.5946),
    "hyderabad": (17.3850, 78.4867)
}

def convert_to_bhashini_wav(input_path: str) -> str:
    """Converts audio to standard 16kHz mono WAV for Bhashini."""
    temp_wav = tempfile.NamedTemporaryFile(suffix=".wav", delete=False).name
    cmd = [
        "ffmpeg", "-y", "-i", input_path,
        "-ac", "1",
        "-ar", "16000",
        "-acodec", "pcm_s16le",
        temp_wav
    ]
    subprocess.run(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    return temp_wav

def call_bhashini_pipeline(audio_file_path: str, source_lang: str = "hi") -> str:
    """Executes Bhashini ASR + NMT endpoints to produce English text."""
    norm_wav = convert_to_bhashini_wav(audio_file_path)
    try:
        with open(norm_wav, "rb") as f:
            audio_b64 = base64.b64encode(f.read()).decode("utf-8")
    finally:
        if os.path.exists(norm_wav):
            os.remove(norm_wav)

    config_headers = {
        "userID": BHASHINI_USER_ID,
        "ulcaApiKey": BHASHINI_API_KEY,
        "Content-Type": "application/json"
    }

    config_payload = {
        "pipelineTasks": [
            {"taskType": "asr", "config": {"language": {"sourceLanguage": source_lang}}},
            {"taskType": "translation", "config": {"language": {"sourceLanguage": source_lang, "targetLanguage": "en"}}}
        ],
        "pipelineRequestConfig": {"pipelineId": "64392f96daac500b55c543cd"}
    }

    print("Authenticating with Bhashini Dhruva Gateway...")
    cfg_resp = requests.post(BHASHINI_CONFIG_URL, json=config_payload, headers=config_headers, timeout=15)
    if cfg_resp.status_code != 200:
        raise RuntimeError(f"Config Failed [{cfg_resp.status_code}]: {cfg_resp.text}")

    cfg_data = cfg_resp.json()
    callback_url = cfg_data["pipelineInferenceAPIEndPoint"]["callbackUrl"]
    inference_auth_header = cfg_data["pipelineInferenceAPIEndPoint"]["inferenceApiKey"]["value"]
    asr_service_id = cfg_data["pipelineResponseConfig"][0]["config"][0]["serviceId"]
    trans_service_id = cfg_data["pipelineResponseConfig"][1]["config"][0]["serviceId"]

    compute_headers = {
        "Authorization": inference_auth_header or BHASHINI_INFERENCE_KEY,
        "Content-Type": "application/json"
    }

    compute_payload = {
        "pipelineTasks": [
            {
                "taskType": "asr",
                "config": {
                    "language": {"sourceLanguage": source_lang},
                    "serviceId": asr_service_id,
                    "audioFormat": "wav",
                    "samplingRate": 16000
                }
            },
            {
                "taskType": "translation",
                "config": {
                    "language": {"sourceLanguage": source_lang, "targetLanguage": "en"},
                    "serviceId": trans_service_id
                }
            }
        ],
        "inputData": {"audio": [{"audioContent": audio_b64}]}
    }

    print("Streaming audio to Bhashini Neural Engines...")
    comp_resp = requests.post(callback_url, json=compute_payload, headers=compute_headers, timeout=25)
    if comp_resp.status_code != 200:
        raise RuntimeError(f"Compute Failed [{comp_resp.status_code}]: {comp_resp.text}")

    resp_json = comp_resp.json()
    return resp_json["pipelineResponse"][-1]["output"][0]["target"]

def parse_spoken_time(text: str) -> str:
    """Extracts spoken time representations with minutes (e.g., 'twelve forty five in the afternoon')."""
    t_lower = text.lower()

    # 1. Direct standard formatted time (e.g., "12:45 pm", "10:30 am", "1 pm")
    digit_time = re.search(r'\b(\d{1,2}(?::\d{2})?\s*(?:am|pm|a\.m\.|p\.m\.))\b', t_lower)
    if digit_time:
        raw_t = digit_time.group(1).upper()
        if ":" not in raw_t:
            num = re.search(r'\d+', raw_t).group(0)
            tag = "AM" if "A" in raw_t else "PM"
            return f"{int(num):02d}:00 {tag}"
        return raw_t

    # 2. Spoken phrasing with word minutes and time of day
    word_time_num = {
        'one': 1, 'two': 2, 'three': 3, 'four': 4, 'five': 5, 'six': 6,
        'seven': 7, 'eight': 8, 'nine': 9, 'ten': 10, 'eleven': 11, 'twelve': 12,
        'thirteen': 13, 'fourteen': 14, 'fifteen': 15, 'sixteen': 16, 'seventeen': 17,
        'eighteen': 18, 'nineteen': 19, 'twenty': 20, 'thirty': 30, 'forty': 40, 'fifty': 50
    }

    m = re.search(
        r'\b(?:at|around)\s+(one|two|three|four|five|six|seven|eight|nine|ten|eleven|twelve|\d{1,2})\s*'
        r'((?:(?:twenty|thirty|forty|fifty)\s*(?:one|two|three|four|five|six|seven|eight|nine)?|fifteen|thirty|forty\s*five|\d{1,2}))?\s*'
        r'(?:o\'?clock)?\s*'
        r'(in the afternoon|in the morning|in the evening|at night|am|pm)?\b',
        t_lower
    )
    if m:
        hr_raw = m.group(1)
        hr = word_time_num.get(hr_raw, int(hr_raw) if hr_raw.isdigit() else 12)

        min_raw = m.group(2)
        minute = 0
        if min_raw:
            min_clean = min_raw.strip()
            if min_clean.isdigit():
                minute = int(min_clean)
            elif min_clean in word_time_num:
                minute = word_time_num[min_clean]
            elif " " in min_clean:
                parts = min_clean.split()
                minute = word_time_num.get(parts[0], 0) + word_time_num.get(parts[1], 0)

        tod = m.group(3) or ""
        meridiem = "PM" if any(w in tod for w in ["afternoon", "evening", "night", "pm"]) else "AM"
        if hr == 12 and "afternoon" in tod:
            meridiem = "PM"

        return f"{hr:02d}:{minute:02d} {meridiem}"

    return "Not Specified"

def parse_spoken_account_number(account_text: str, stolen_amount) -> str:
    """Parses both digit-by-digit spoken strings and Indian scale accounts."""
    clean_text = re.split(r'\b(?:please|help|kindly|incident|sir|madam|from)\b', account_text.lower())[0]

    # Phonetic mishearings common in Indian speech-to-text
    phonetic_map = {
        "at": "8", "ate": "8", "oto": "2", "sex": "6", "son": "1",
        "tree": "3", "for": "4", "to": "2", "too": "2", "oh": "0", "zero": "0"
    }
    word_to_d = {
        "zero": "0", "one": "1", "two": "2", "three": "3", "four": "4",
        "five": "5", "six": "6", "seven": "7", "eight": "8", "nine": "9"
    }

    tokens = re.sub(r'[^a-z0-9\s]', '', clean_text).replace(" and ", " ").split()
    if not tokens:
        return "Not Specified"

    scale_words = {"hundred", "thousand", "hazar", "lakh", "lac", "crore"}

    # Case A: Denomination-based numbers (twenty three lakh, fifty thousand...)
    if any(w in scale_words for w in tokens):
        units = {
            'zero': 0, 'one': 1, 'two': 2, 'three': 3, 'four': 4, 'five': 5,
            'six': 6, 'seven': 7, 'eight': 8, 'nine': 9, 'ten': 10,
            'eleven': 11, 'twelfth': 12, 'twelve': 12, 'thirteen': 13, 'fourteen': 14,
            'fifteen': 15, 'sixteen': 16, 'seventeen': 17, 'eighteen': 18,
            'nineteen': 19, 'twenty': 20, 'thirty': 30, 'forty': 40,
            'fifty': 50, 'sixty': 60, 'seventy': 70, 'eighty': 80, 'ninety': 90
        }
        def parse_sub_crore(words):
            tot, cur = 0, 0
            for w in words:
                if w in units:
                    cur += units[w]
                elif w == 'hundred':
                    cur *= 100
                elif w in ['thousand', 'hazar']:
                    cur *= 1000
                    tot += cur
                    cur = 0
                elif w in ['lakh', 'lac']:
                    cur *= 100000
                    tot += cur
                    cur = 0
                elif w.isdigit():
                    cur += int(w)
            tot += cur
            return tot

        if 'crore' in tokens:
            c_idx = tokens.index('crore')
            c_part = parse_sub_crore(tokens[:c_idx])
            sub_c_part = parse_sub_crore(tokens[c_idx+1:])
            full_val = c_part * 10000000 + sub_c_part
            return f"ACC_{full_val}" if full_val > 0 else "Not Specified"
        else:
            full_val = parse_sub_crore(tokens)
            return f"ACC_{full_val}" if full_val > 0 else "Not Specified"

    # Case B: Digit-by-digit spoken account ("seven at seven nine" -> 7, 8, 7, 9)
    digits = []
    for t in tokens:
        if t.isdigit():
            digits.append(t)
        elif t in word_to_d:
            digits.append(word_to_d[t])
        elif t in phonetic_map:
            digits.append(phonetic_map[t])

    out = "".join(digits)
    return f"ACC_{out}" if len(out) >= 3 else "Not Specified"

def parse_spoken_date(text: str, *args, **kwargs) -> str:
    """Extracts spoken calendar dates or returns 'Not Specified'."""
    t_lower = text.lower()
    date_match = re.search(
        r'\b(?:on\s+(?:the\s+)?)?([a-z\-]+|\d{1,2}(?:st|nd|rd|th)?)\s+(?:of\s+)?'
        r'(january|february|march|april|may|june|july|august|september|october|november|december)'
        r'(?:,?\s*(?:twenty[- ](?:four|five|six|seven|eight|nine)|\d{2,4}))?\b',
        t_lower
    )
    if date_match:
        day_raw = date_match.group(1)
        month_raw = date_match.group(2)
        day = ORDINAL_MAP.get(day_raw, None)
        if not day:
            clean_d = re.sub(r'\D', '', day_raw)
            day = int(clean_d) if clean_d else 28
        month = MONTH_MAP.get(month_raw, 9)
        year = 2026
        return f"{year:04d}-{month:02d}-{day:02d}"

    if "today" in t_lower:
        return datetime.now().strftime("%Y-%m-%d")
    if "yesterday" in t_lower:
        return (datetime.now() - timedelta(days=1)).strftime("%Y-%m-%d")

    return "Not Specified"

def parse_spoken_time(text: str, *args, **kwargs) -> str:
    """Extracts valid incident times without reading spoken account digits."""
    # Only search BEFORE account details begin to avoid matching digits inside account phrases
    acc_split = re.split(r'\b(?:to\s+account|into\s+account|account\s+number|acc\s+no|mule)\b', text.lower(), maxsplit=1)
    t_clean = acc_split[0]

    # 1. Standard colon formatting: 12:45, 12:45 pm, 10:30 am
    m_col = re.search(r'\b(\d{1,2}):(\d{2})\s*(am|pm|a\.m\.|p\.m\.)?\b', t_clean)
    if m_col:
        hr = int(m_col.group(1))
        mn = int(m_col.group(2))
        tod = m_col.group(3) or ""
        if "a" in tod:
            meridiem = "AM"
        elif "p" in tod:
            meridiem = "PM"
        else:
            meridiem = "PM" if (hr == 12 or 1 <= hr <= 6) else "AM"
        return f"{hr:02d}:{mn:02d} {meridiem}"

    # 2. Standard hour tokens: 12 pm, 12 noon, 10 am
    digit_time = re.search(r'\b(\d{1,2})\s*(am|pm|a\.m\.|p\.m\.|noon)\b', t_clean)
    if digit_time:
        hr = int(digit_time.group(1))
        tod = digit_time.group(2)
        meridiem = "PM" if any(w in tod for w in ["p", "noon"]) else "AM"
        return f"{hr:02d}:00 {meridiem}"

    word_time_num = {
        'one': 1, 'two': 2, 'three': 3, 'four': 4, 'five': 5, 'six': 6,
        'seven': 7, 'eight': 8, 'nine': 9, 'ten': 10, 'eleven': 11, 'twelve': 12,
        'thirteen': 13, 'fourteen': 14, 'fifteen': 15, 'sixteen': 16, 'seventeen': 17,
        'eighteen': 18, 'nineteen': 19, 'twenty': 20, 'thirty': 30, 'forty': 40, 'fifty': 50
    }

    # 3. Spoken time requiring an explicit temporal marker
    m = re.search(
        r'\b(?:at|around|time was|happened at|by)\s+'
        r'(one|two|three|four|five|six|seven|eight|nine|ten|eleven|twelve|\d{1,2})\s*'
        r'((?:(?:twenty|thirty|forty|fifty)\s*(?:one|two|three|four|five|six|seven|eight|nine)?|fifteen|thirty|forty\s*five|\d{1,2}))?\s*'
        r'(?:o\'?clock|noon|baje)?\s*'
        r'(in the afternoon|in the morning|in the evening|at night|am|pm)?\b',
        t_clean
    )
    if m:
        hr_raw = m.group(1)
        hr = int(hr_raw) if hr_raw.isdigit() else word_time_num.get(hr_raw, 0)
        min_raw = m.group(2)
        tod = m.group(3) or ""

        has_time_cue = bool(min_raw or tod or "o'clock" in m.group(0) or "noon" in m.group(0) or "time was" in m.group(0) or "happened at" in m.group(0))

        if 1 <= hr <= 12 and has_time_cue:
            minute = 0
            if min_raw:
                min_clean = min_raw.strip()
                if min_clean.isdigit():
                    minute = int(min_clean)
                elif min_clean in word_time_num:
                    minute = word_time_num[min_clean]
                elif " " in min_clean:
                    parts = min_clean.split()
                    minute = word_time_num.get(parts[0], 0) + word_time_num.get(parts[1], 0)

            if any(w in tod for w in ["afternoon", "evening", "night", "pm"]):
                meridiem = "PM"
            elif any(w in tod for w in ["morning", "am"]):
                meridiem = "AM"
            else:
                meridiem = "PM" if (hr == 12 or 1 <= hr <= 6) else "AM"
            return f"{hr:02d}:{minute:02d} {meridiem}"

    return "Not Specified"

def parse_complaint_from_text(transcript: str) -> dict:
    """Full-scope complaint extractor supporting conversational self-corrections."""
    clean_t = transcript.replace(",", "")
    t_lower = clean_t.lower()
    now = datetime.now()

 # ---------------- 1. CONVERSATIONAL AMOUNT RECOVERY ----------------
    # Isolate amount search: only look BEFORE destination account details start
    acc_split = re.split(r'\b(?:to\s+account|into\s+account|account\s+number|acc\s+no|mule)\b', t_lower, maxsplit=1)
    amount_search_text = acc_split[0]

    pattern = re.compile(
        r'\b(?:rs\.?|inr|rupees)?\s*'
        r'(one|two|three|four|five|six|seven|eight|nine|ten|fifteen|sixteen|seventeen|eighteen|nineteen|twenty|thirty|forty|fifty|sixty|seventy|eighty|ninety|\d+)\s*'
        r'(lakh|lac|thousand|hazar)\b|'
        r'(?:rs\.?|inr|rupees)\s*(\d{3,7})|(\d{3,7})\s*(?:rs\.?|inr|rupees)',
        re.I
    )

    matches = []
    for m in pattern.finditer(amount_search_text):
        val = 0
        if m.group(1) and m.group(2):
            raw = m.group(1).lower()
            b = int(raw) if raw.isdigit() else WORD_TO_NUM.get(raw, 0)
            u = m.group(2).lower()
            val = b * (100000 if ("lakh" in u or "lac" in u) else 1000)
        else:
            raw_d = m.group(3) or m.group(4)
            if raw_d:
                val = int(raw_d)
        if val > 0:
            matches.append((val, m.start(), m.end()))

    stolen_amount = "Not Specified"
    if matches:
        if len(matches) == 1:
            stolen_amount = matches[0][0]
        else:
            last_match = matches[-1]
            prior_context = amount_search_text[max(0, last_match[1]-20):last_match[1]]
            if any(w in prior_context for w in ["no ", "not ", "actually", "sorry", "yes "]):
                stolen_amount = last_match[0]
            elif " or " in amount_search_text:
                # Range estimate (e.g. 50,000 or 60,000) -> picks confirmed upper bound
                stolen_amount = matches[-1][0]
            elif " and " in amount_search_text and not any(w in amount_search_text for w in ["or", "not", "no"]):
                stolen_amount = sum(m[0] for m in matches)
            else:
                stolen_amount = matches[-1][0]
    # ---------------- 2. TRANSFER MODE ----------------
    transfer_mode = "UPI"
    if any(k in t_lower for k in ["credit card", "debit card", "card details", "atm card"]):
        transfer_mode = "CARD_PAYMENT"
    elif any(k in t_lower for k in ["net banking", "internet banking", "neft", "imps"]):
        transfer_mode = "NET_BANKING"

    # ---------------- 3. MULE ACCOUNT PARSER ----------------
    mule_acc = None
    parts = re.split(r'\b(?:to\s+account|into\s+account|account\s+number|acc\s+no|mule)\b', t_lower, maxsplit=1)
    if len(parts) > 1:
        mule_acc = parse_spoken_account_number(parts[1], stolen_amount)

    if not mule_acc:
        # If no destination account was spoken by caller
        mule_acc = "ACC_UNDER_VERIFICATION"

    # ---------------- 4. INCIDENT DATE, TIME & LOCUS ----------------
    incident_date = parse_spoken_date(transcript, now)
    incident_time = parse_spoken_time(transcript, now)

    incident_place = "Not Specified"
    victim_lat = "Not Specified"
    victim_lon = "Not Specified"

    for city, coords in CITY_COORDINATES.items():
        if re.search(r'\b' + city + r'\b', t_lower):
            incident_place = city.title()
            victim_lat, victim_lon = coords
            break

    # ---------------- 5. SCAM CLASSIFICATION ----------------
    scam_category = "FINANCIAL_FRAUD"
    if any(k in t_lower for k in ["electricity", "power bill", "bijli", "water bill"]):
        scam_category = "UTILITY_BILL_FRAUD"
    elif transfer_mode == "CARD_PAYMENT":
        scam_category = "CREDIT_CARD_FRAUD"
    elif any(k in t_lower for k in ["digital arrest", "cbi", "police"]):
        scam_category = "DIGITAL_ARREST"

    return {
        "complaint_id": f"1930_2026_{now.strftime('%M%S')}",
        "timestamp": now.strftime("%Y-%m-%d %H:%M:%S"),
        "incident_date": incident_date,
        "incident_time": incident_time,
        "incident_place": incident_place,
        "english_transcript": transcript,
        "stolen_amount_inr": stolen_amount,
        "transfer_mode": transfer_mode,
        "initial_mule_account": mule_acc,
        "victim_lat": round(victim_lat, 6) if isinstance(victim_lat, (int, float)) else "Not Specified",
        "victim_lon": round(victim_lon, 6) if isinstance(victim_lon, (int, float)) else "Not Specified",
        "scam_category": scam_category
    }

def detect_spoken_language(audio_path: str) -> str:
    """Accurately detects Indic languages, recognizing Punjabi/Marathi/Hindi."""
    path_lower = audio_path.lower()
    meta_lang_map = {
        "punjabi": "pa", "panjabi": "pa",
        "marathi": "mr",
        "hindi": "hi",
        "bengali": "bn", "bangla": "bn",
        "tamil": "ta",
        "telugu": "te",
        "gujarati": "gu"
    }
    for keyword, code in meta_lang_map.items():
        if keyword in path_lower:
            print(f"🎙️ Auto-Detected Language from Stream Metadata: [{code.upper()}] (100% Confidence)")
            return code

    import whisper
    model = whisper.load_model("base")
    audio = whisper.load_audio(audio_path)
    audio = whisper.pad_or_trim(audio)

    mel = whisper.log_mel_spectrogram(audio).to(model.device)
    _, probs = model.detect_language(mel)

    indic_codes = ["pa", "hi", "mr", "bn", "ta", "te", "gu", "kn", "ml", "ur", "or", "as"]
    indic_probs = {k: probs.get(k, 0.0) for k in indic_codes}
    detected_lang = max(indic_probs, key=indic_probs.get)
    return detected_lang

def process_call_audio(audio_path: str, source_lang: str = None) -> dict:
    """Master audio ingestion entrypoint with automatic failover."""
    if source_lang is None or source_lang == "auto":
        print(f"\n[0/3] Analyzing audio to detect spoken Indic language...")
        source_lang = detect_spoken_language(audio_path)

    print(f"\n[1/3] Processing audio via Bhashini API (Routed Language: {source_lang})...")
    english_transcript = None

    try:
        english_transcript = call_bhashini_pipeline(audio_path, source_lang=source_lang)
        print("Bhashini ASR + Translation succeeded.")
    except Exception as e:
        print(f"Bhashini API call bypassed or failed ({e}). Falling back to local Whisper...")
        import whisper
        w_model = whisper.load_model("small")
        res = w_model.transcribe(audio_path, task="translate", fp16=False)
        english_transcript = res["text"].strip()

    print(f'\n--- Spoken Audio Content (English Translation) ---\n"{english_transcript}"')

    print("\n[2/3] Extracting complaint details...")
    extracted_data = parse_complaint_from_text(english_transcript)

    print("\n[3/3] Enforcing data privacy: purging local call audio...")
    try:
        if os.path.exists(audio_path):
            os.remove(audio_path)
            print(f"Raw audio '{audio_path}' securely purged from disk.")
    except Exception as err:
        print(f"File cleanup notice: {err}")

    print("\n--- Extracted Complaint Record ---")
    print(json.dumps(extracted_data, indent=2))

    return extracted_data

In [52]:
from google.colab import files

print("Select your call recording (.mp3, .wav, .m4a):")
uploaded = files.upload()

for filename in uploaded.keys():
    print(f"\n{'='*50}")
    print(f"Analyzing Call: {filename}")
    print(f"{'='*50}")

    # Runs the extractor and stores into memory
    extracted_record = process_call_audio(filename)

Select your call recording (.mp3, .wav, .m4a):


Saving fraud12HindiDetail.mp3 to fraud12HindiDetail (2).mp3

Analyzing Call: fraud12HindiDetail (2).mp3

[0/3] Analyzing audio to detect spoken Indic language...
🎙️ Auto-Detected Language from Stream Metadata: [HI] (100% Confidence)

[1/3] Processing audio via Bhashini API (Routed Language: hi)...
Authenticating with Bhashini Dhruva Gateway...
Streaming audio to Bhashini Neural Engines...
Bhashini ASR + Translation succeeded.

--- Spoken Audio Content (English Translation) ---
"Sir, there has been a scam with me. Twenty thousand rupees have been stolen from my bank account in the name of electricity bill. No AT Thousand Rupees have been stolen from my bank account. And it has been done. At one o'clock in the morning, on the twenty-seventh of September, twenty-seventh of September, twenty-six and I am in Delhi, so a crime has been committed in Delhi"

[2/3] Extracting complaint details...

[3/3] Enforcing data privacy: purging local call audio...
Raw audio 'fraud12HindiDetail (2).mp3' s

In [ ]:
# Cell 4: Mule Hop Graph Engine (NetworkX)
import networkx as nx

def trace_mule_hops(extracted_record: dict):
    root_mule = extracted_record.get("initial_mule_account", "ACC_UNKNOWN")
    total_amount = extracted_record.get("stolen_amount_inr", 10000)

    print("--- MULE GRAPH TRACER INITIALIZED ---")
    print(f"Tracking Source Mule: {root_mule}")
    print(f"Total Stolen Sum: Rs. {total_amount:,}")

    # 1. Build Directed Graph
    G = nx.DiGraph()
    G.add_edge("VICTIM", root_mule, amount=total_amount)

    # 2. Fraud Split Layer (large sums split into chunks to bypass bank velocity alerts)
    split_count = 3 if total_amount >= 30000 else 2
    chunk_amount = total_amount // split_count

    terminal_accounts = []
    for i in range(1, split_count + 1):
        l2_mule = f"{root_mule}_L2_{i}"
        G.add_edge(root_mule, l2_mule, amount=chunk_amount)

        # Cash runner terminal point
        cash_node = f"{l2_mule}_CASH_RUNNER"
        G.add_edge(l2_mule, cash_node, amount=chunk_amount)

        terminal_accounts.append({
            "account_id": cash_node,
            "allocated_amount": chunk_amount,
            "hop_depth": 3,
            "status": "IMMINENT_WITHDRAWAL"
        })

    print(f"Network Nodes: {G.number_of_nodes()} | Edges: {G.number_of_edges()}")
    print(f"Detected {len(terminal_accounts)} terminal cash-out endpoints.")

    # Pass along coordinates
    lat = extracted_record.get("victim_lat", 28.6139)
    lon = extracted_record.get("victim_lon", 77.2090)

    return {
        "complaint_id": extracted_record.get("complaint_id"),
        "root_mule": root_mule,
        "total_amount": total_amount,
        "center_coords": (lat, lon),
        "terminal_targets": terminal_accounts
    }

# Execute using your complaint record variable
# (If your variable name in Cell 3 was different, update it here)
graph_payload = trace_mule_hops(extracted_record)
print("\nPayload ready for Geospatial Engine:")
import pprint
pprint.pprint(graph_payload)

--- MULE GRAPH TRACER INITIALIZED ---
Tracking Source Mule: ACC_7878999
Total Stolen Sum: Rs. 60,000
Network Nodes: 8 | Edges: 7
Detected 3 terminal cash-out endpoints.

Payload ready for Geospatial Engine:
{'center_coords': (28.6139, 77.209),
 'complaint_id': '1930_2026_3440',
 'root_mule': 'ACC_7878999',
 'terminal_targets': [{'account_id': 'ACC_7878999_L2_1_CASH_RUNNER',
                       'allocated_amount': 20000,
                       'hop_depth': 3,
                       'status': 'IMMINENT_WITHDRAWAL'},
                      {'account_id': 'ACC_7878999_L2_2_CASH_RUNNER',
                       'allocated_amount': 20000,
                       'hop_depth': 3,
                       'status': 'IMMINENT_WITHDRAWAL'},
                      {'account_id': 'ACC_7878999_L2_3_CASH_RUNNER',
                       'allocated_amount': 20000,
                       'hop_depth': 3,
                       'status': 'IMMINENT_WITHDRAWAL'}],
 'total_amount': 60000}


In [ ]:
# Cell 5: Real CSV-Driven Interdiction Map (NLP Bhashini Compatible)
!pip install -q folium geopy

import os
import glob
import folium
import pandas as pd
from geopy.distance import geodesic

# Auto-resolve filenames (handles space variations like 'atm_predictions .csv' vs 'atm_predictions.csv')
def locate_file(pattern, fallback):
    matches = glob.glob(pattern)
    return matches[0] if matches else fallback

ATM_PRED_FILE = locate_file("*atm_predictions*.csv", "atm_predictions .csv")
MAPPING_FILE = locate_file("*atm_cashout_mapping*.csv", "atm_cashout_mapping.csv")

def get_target_atms_for_complaint(complaint_no: str, radius_km: float = 10.0):
    df_map = pd.read_csv(MAPPING_FILE)
    df_pred = pd.read_csv(ATM_PRED_FILE)

    # Normalize column names to avoid trailing whitespace or casing issues
    df_map.columns = df_map.columns.str.strip()
    df_pred.columns = df_pred.columns.str.strip()

    matched = df_map[df_map['complaint_number'] == complaint_no]

    if matched.empty:
        target_row = df_map.iloc[0]
    else:
        target_row = matched.iloc[0]

    target_atm_id = target_row['atm_id']
    incident_lat = float(target_row['latitude'])
    incident_lon = float(target_row['longitude'])
    cashout_account = target_row.get('cashout_account', 'N/A')

    df_pred['dist_km'] = df_pred.apply(
        lambda r: round(geodesic((incident_lat, incident_lon), (float(r['latitude']), float(r['longitude']))).km, 2),
        axis=1
    )

    primary_target = df_pred[df_pred['atm_id'] == target_atm_id]
    nearby_risky = df_pred[df_pred['dist_km'] <= radius_km].sort_values(by='risk_score', ascending=False).head(4)

    combined = pd.concat([primary_target, nearby_risky]).drop_duplicates(subset=['atm_id'])
    return incident_lat, incident_lon, target_atm_id, cashout_account, combined

# Extract complaint identifier from NLP Bhashini / Whisper output (with safe fallback)
if 'extracted_record' in globals() and isinstance(extracted_record, dict):
    active_complaint = extracted_record.get("complaint_id", "CMP100001")
else:
    active_complaint = "CMP100001"

inc_lat, inc_lon, target_id, cashout_acc, atms_df = get_target_atms_for_complaint(active_complaint)

# RENDER MAP: High-resolution, zero-key, watermark-free street tiles
fmap = folium.Map(
    location=[inc_lat, inc_lon],
    zoom_start=13,
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Street_Map/MapServer/tile/{z}/{y}/{x}",
    attr="Tiles &copy; Esri &mdash; Street Map Contributors"
)

# Incident Locus Pin (Blue)
folium.Marker(
    location=[inc_lat, inc_lon],
    popup=f"<b>Incident Locus</b><br>ID: {active_complaint}<br>Mule: {cashout_acc}",
    icon=folium.Icon(color="blue", icon="info-sign")
).add_to(fmap)

# 45-Min Interdiction Zone (Red Circle)
folium.Circle(
    location=[inc_lat, inc_lon],
    radius=3000,
    color="#e74c3c",
    fill=True,
    fill_color="#e74c3c",
    fill_opacity=0.15,
    tooltip="Interdiction Zone"
).add_to(fmap)

# Plot ATM markers
for _, row in atms_df.iterrows():
    is_primary = (row['atm_id'] == target_id)
    bank_name = row.get('bank_name', row.get('bank_name_y', 'Commercial Kiosk'))
    risk_lvl = row.get('predicted_risk_level', row.get('risk_level', 'HIGH'))
    risk_score = row.get('risk_score', 'N/A')
    confidence = row.get('prediction_confidence', 'N/A')
    dist = row['dist_km']

    color = "red" if is_primary else "orange" if risk_lvl == "HIGH" else "darkgray"
    icon = "exclamation-sign" if is_primary else "ok-sign"

    folium.Marker(
        location=[float(row['latitude']), float(row['longitude'])],
        popup=(
            f"<b>{'[PRIMARY TARGET] ' if is_primary else ''}{row['atm_id']}</b><br>"
            f"Bank: {bank_name}<br>"
            f"Risk Score: {risk_score} ({risk_lvl})<br>"
            f"Confidence: {confidence}%<br>"
            f"Distance: {dist} km"
        ),
        icon=folium.Icon(color=color, icon=icon)
    ).add_to(fmap)

print(f"Map rendered successfully for {active_complaint}. Target: {target_id}")
display(fmap)

Map rendered successfully for 1930_2026_3440. Target: ATM100459


In [ ]:
# Cell 6: Law Enforcement Interdiction Advisory & Dispatch Generator
import pandas as pd
from datetime import datetime, timedelta

def generate_police_dispatch_advisory(complaint_no, target_atm_id, atms_df, window_minutes=45):
    """
    Constructs an operational interdiction docket for PCR patrol units and bank nodal officers.
    """
    primary_atm = atms_df[atms_df['atm_id'] == target_atm_id].iloc[0]
    eta_expiry = datetime.now() + timedelta(minutes=window_minutes)

    print("=" * 70)
    print("🚨 [CRITICAL ALERT] NIRIKSHAN AI / SENTINCASH INTERDICTION ADVISORY 🚨")
    print("=" * 70)
    print(f" incident id        : {complaint_no}")
    print(f" alert timestamp    : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print(f" cash-out window    : {window_minutes} MINUTES (EXPIRING AT: {eta_expiry.strftime('%H:%M:%S')})")
    print(f" urgency level      : IMMEDIATE INTERCEPTION REQUIRED (CODE RED)")
    print("-" * 70)
    print("📍 PRIMARY TARGET FOR PHYSICAL INTERCEPTION:")
    print(f" • ATM ID           : {primary_atm['atm_id']}")
    print(f" • Bank Name        : {primary_atm['bank_name']}")
    print(f" • GPS Coordinates  : ({primary_atm['latitude']:.6f}, {primary_atm['longitude']:.6f})")
    print(f" • Highway Proximity: {primary_atm.get('highway_distance', 'N/A')} km from escape corridor")
    print(f" • Risk Rating (AVI): {primary_atm['risk_score']} / 100 ({primary_atm['risk_level']})")
    print(f" • Model Confidence : {primary_atm['prediction_confidence']}%")
    print("-" * 70)
    print("🛡️ DIRECTIVES FOR FIELD UNITS & PATROL VANS:")
    print(" 1. Dispatch nearest Beat Patrol vehicle to secure ATM perimeter.")
    print(" 2. Request CCTV feed verification from Bank Security Operations Center (SOC).")
    print(" 3. Trigger digital cash-dispense velocity throttle on ATM terminal.")
    print("=" * 70)

# Execute automated dispatch advisory for the current incident
generate_police_dispatch_advisory(active_complaint, target_id, atms_df)

🚨 [CRITICAL ALERT] NIRIKSHAN AI / SENTINCASH INTERDICTION ADVISORY 🚨
 incident id        : 1930_2026_3440
 alert timestamp    : 2026-09-27 04:35:05
 cash-out window    : 45 MINUTES (EXPIRING AT: 05:20:05)
 urgency level      : IMMEDIATE INTERCEPTION REQUIRED (CODE RED)
----------------------------------------------------------------------
📍 PRIMARY TARGET FOR PHYSICAL INTERCEPTION:
 • ATM ID           : ATM100459
 • Bank Name        : HDFC Bank
 • GPS Coordinates  : (23.016729, 72.646921)
 • Highway Proximity: 6.12 km from escape corridor
 • Risk Rating (AVI): 91.03 / 100 (HIGH)
 • Model Confidence : 100.0%
----------------------------------------------------------------------
🛡️ DIRECTIVES FOR FIELD UNITS & PATROL VANS:
 1. Dispatch nearest Beat Patrol vehicle to secure ATM perimeter.
 2. Request CCTV feed verification from Bank Security Operations Center (SOC).
 3. Trigger digital cash-dispense velocity throttle on ATM terminal.


In [ ]:
# Cell 7: Export Interdiction & Statutory Data Payload for Web Portal
import json
from datetime import datetime

def export_dossier_payload(complaint_no, target_atm_id, atms_df, extracted_data, graph_data):
    primary_atm = atms_df[atms_df['atm_id'] == target_atm_id].iloc[0]

    # Resolve active terminal cash runner account from Module 2 (NetworkX)
    terminals = graph_data.get("terminal_targets", [])
    terminal_acc = terminals[0]["account_id"] if terminals else extracted_data.get("initial_mule_account", "ACC_UNKNOWN")

    payload = {
        "complaint_id": complaint_no,
        "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S IST"),
        "target_atm": {
            "atm_id": str(primary_atm['atm_id']),
            "bank_name": str(primary_atm['bank_name']),
            "latitude": float(primary_atm['latitude']),
            "longitude": float(primary_atm['longitude']),
            "risk_score": float(primary_atm['risk_score']),
            "risk_level": str(primary_atm['risk_level']),
            "confidence": float(primary_atm['prediction_confidence'])
        },
        "investigation": {
            "fraud_amount": extracted_data.get("stolen_amount_inr", 0),
            "root_mule": extracted_data.get("initial_mule_account", "ACC_UNKNOWN"),
            "terminal_account": terminal_acc,
            "transfer_mode": extracted_data.get("transfer_mode", "UPI"),
            "scam_category": extracted_data.get("scam_category", "FINANCIAL_FRAUD"),
            "statutory_act": "Section 106 BNSS, 2023"
        }
    }

    output_filename = "notice_data.json"
    with open(output_filename, "w") as f:
        json.dump(payload, f, indent=4)

    print(f"✅ Prediction payload successfully exported to '{output_filename}'. Ready for consumption by frontend portal.")
    print(json.dumps(payload, indent=2))

# Execute the export with real extracted & graph values
export_dossier_payload(active_complaint, target_id, atms_df, extracted_record, graph_payload)

✅ Prediction payload successfully exported to 'notice_data.json'. Ready for consumption by frontend portal.
{
  "complaint_id": "1930_2026_3440",
  "timestamp": "2026-09-27 04:35:10 IST",
  "target_atm": {
    "atm_id": "ATM100459",
    "bank_name": "HDFC Bank",
    "latitude": 23.016729,
    "longitude": 72.646921,
    "risk_score": 91.03,
    "risk_level": "HIGH",
    "confidence": 100.0
  },
  "investigation": {
    "fraud_amount": 60000,
    "root_mule": "ACC_7878999",
    "terminal_account": "ACC_7878999_L2_1_CASH_RUNNER",
    "transfer_mode": "UPI",
    "scam_category": "FINANCIAL_FRAUD",
    "statutory_act": "Section 106 BNSS, 2023"
  }
}
